# PDDL Attack Path Evaluation
Evaluate generated PDDL attack paths: solvability, syntax, and semantic quality.

## 1. Environment Setup (imports, constnats, global variables)

In [2]:
import sys
import os
import re
import json
from dataclasses import dataclass
from pathlib import Path
import numpy as np

from cve2pddlap.core.data_loader import load_few_shot_pool
from cve2pddlap.evaluation import create_ff_checker, create_enhsp_checker
from cve2pddlap.evaluation.problem_pddl_generator import generate_problem

import torch
from sentence_transformers import SentenceTransformer, util as st_util
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline as hf_pipeline

from cve2pddlap.llm_providers.remote.openai_compat import QwenProvider
from openai import OpenAI

from jinja2 import Environment, FileSystemLoader

from cve2pddlap.core.data_loader import load_few_shot_pool


# Project root (relative — works from notebooks/attack_paths/)
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..', '..'))
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

# Data paths and file names
DATASET_PATH = os.path.join(PROJECT_ROOT, 'resources', 'data', 'CVE-PDDL-NNL-ReAP')
TARGET_POOL_FILE = os.path.join(PROJECT_ROOT, 'resources', 'data', 'target_pool.json')
GENERATED_DOMAIN_DIR = os.path.join(PROJECT_ROOT, 'generated_domain')
EVAL_SET_DIR = os.path.join(GENERATED_DOMAIN_DIR, 'eval_set')
PROMPTS_PATH = os.path.join(PROJECT_ROOT, 'resources', 'prompt', 'evaluation')
AP_PATTERN = re.compile(r'^AP\d+$')
DOMAIN_FILE = 'domain.pddl'
PROBLEM_FILE = 'problem.pddl'

# Models (small defaults — replace with preferred models)
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
LLM_MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

# Generation parameters
SEED = 42
TEMPERATURE = 0.0
TOP_K = 1


@dataclass(frozen=True)
class EvaluationFlags:
    syntax_check: bool = True
    embedding_intrinsic: bool = True
    embedding_extrinsic: bool = True
    llm_intrinsic: bool = True
    llm_extrinsic: bool = True


eval_flags = EvaluationFlags()

## 2. Load data, Prompts, LLM (tokenizer and embedding model)

In [3]:
few_shot_pool = load_few_shot_pool(DATASET_PATH)
print(f'Reference examples: {len(few_shot_pool)}')
for ex in few_shot_pool[:5]:
    print(f'  {ex.key}')
print('  ...')
print(f'Generated domain dir to be evaluated: {os.path.relpath(EVAL_SET_DIR)}')

Reference examples: 55
  CVE-2022-1471 / AP1
  CVE-2022-40149 / AP1
  CVE-2022-40149 / AP2
  CVE-2022-40150 / AP1
  CVE-2022-40150 / AP2
  ...
Generated domain dir to be evaluated: ../../generated_domain/eval_set


In [4]:
def load_target_pool(target_pool_file):
    """Load CVE descriptions from target_pool.json. Returns {cve_id: description}."""
    with open(target_pool_file, encoding='utf-8') as f:
        pool = json.load(f)
    return {entry['cve_id']: entry['description'] for entry in pool}


def load_dataset(data_path, cve_descriptions):
    """Load all CVEs with their descriptions (from target_pool) and attack path PDDL files."""
    dataset = []
    for cve_dir in sorted(Path(data_path).iterdir()):
        if not cve_dir.is_dir():
            continue
        cve_id = cve_dir.name
        description = cve_descriptions.get(cve_id)
        if description is None:
            continue
        attack_paths = []
        for ap_dir in sorted(cve_dir.iterdir()):
            if not ap_dir.is_dir() or not AP_PATTERN.match(ap_dir.name):
                continue
            domain_file = ap_dir / DOMAIN_FILE
            problem_file = ap_dir / PROBLEM_FILE
            if domain_file.exists() and problem_file.exists():
                attack_paths.append({
                    'ap_id': ap_dir.name,
                    'domain': domain_file.read_text(encoding='utf-8').strip(),
                    'problem': problem_file.read_text(encoding='utf-8').strip(),
                })
        dataset.append({
            'cve_id': cve_id,
            'description': description,
            'attack_paths': attack_paths,
        })
    return dataset


def load_prompts(prompts_path):
    """Load Jinja2 evaluation prompt templates."""
    return Environment(loader=FileSystemLoader(prompts_path))


def load_embedding_model(model_name):
    """Load a SentenceTransformer bi-encoder model."""
    return SentenceTransformer(model_name)


def load_llm(model_name):
    """Load a HuggingFace causal LLM with its tokenizer."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.float16,
        device_map='auto',
    )
    gen = hf_pipeline(
        'text-generation',
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=512,
        do_sample=False,
    )
    return gen, tokenizer


cve_descriptions = load_target_pool(TARGET_POOL_FILE)
dataset = load_dataset(DATASET_PATH, cve_descriptions)
prompt_env = load_prompts(PROMPTS_PATH)

embedding_model = None
if eval_flags.embedding_intrinsic or eval_flags.embedding_extrinsic:
    embedding_model = load_embedding_model(EMBEDDING_MODEL_NAME)

llm, tokenizer = None, None
if eval_flags.llm_intrinsic or eval_flags.llm_extrinsic:
    llm, tokenizer = load_llm(LLM_MODEL_NAME)

Device set to use cpu
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


## 3. Evaluation

### 3.1 Syntactic

#### 3.1.1 Intrinsic: PDDL AP Syntax and Solvability Check

Block 1: The function that tests if the code is syntactically correct

In [5]:
ff = create_ff_checker()
enhsp = create_enhsp_checker()

def check_pddl_ap_syntax(domain_str, problem_save_path=None):
    """Generate problem from domain, then check syntax (ENHSP) and solvability (FF).
    If problem_save_path is provided, saves problem.pddl to that path."""
    problem_str = generate_problem(domain_str, save_path=problem_save_path)
    r_ff = ff.check_from_string(domain_str, problem_str)
    r_enhsp = enhsp.check_from_string(domain_str, problem_str)
    return r_enhsp.success, r_ff.solvable, r_ff.plan, r_enhsp.error or r_ff.error

Block 2: Call the function with the code from the data set

In [6]:
#Reference domains (from dataset, problem already exists but we regenerate for consistency)
# reference_ap_syntax_results = []
# for entry in dataset:
#      for ap in entry['attack_paths']:
#          enhsp_ok, ff_solvable, plan, err = check_pddl_ap_syntax(ap['domain'])
#          result = {
#              'source': 'reference',
#              'cve_id': entry['cve_id'],
#              'ap_id': ap['ap_id'],
#              'syntax_ok': enhsp_ok,
#              'solvable': ff_solvable,
#              'ap': plan,
#              'error': err,
#          }
#          reference_ap_syntax_results.append(result)
#          print(f"Reference: {entry['cve_id']}/{ap['ap_id']}  syntax:{enhsp_ok}  solvable:{ff_solvable}")
#          if plan:
#              for i, action in enumerate(plan):
#                  print(f"  {i}: {action}")

# Generated domains (from eval_set/<CVE-ID>/llm_config/domain.pddl)
# Problem saved to eval_set/C<CVE-ID>/llm_config/problem.pddl
generated_ap_syntax_results = []
eval_set_path = Path(EVAL_SET_DIR)
for cve_dir in sorted(eval_set_path.iterdir()):
    if not cve_dir.is_dir():
        continue
    for config_dir in sorted(cve_dir.iterdir()):
        if not config_dir.is_dir():
            continue
        domain_file = config_dir / 'domain.pddl'
        if not domain_file.exists():
            continue
        domain_str = domain_file.read_text(encoding='utf-8')
        problem_save = str(config_dir / 'problem.pddl')
        enhsp_ok, ff_solvable, plan, err = check_pddl_ap_syntax(domain_str, problem_save_path=problem_save)
        result = {
            'source': 'generated',
            'cve_id': cve_dir.name,
            'config': config_dir.name,
            'syntax_ok': enhsp_ok,
            'solvable': ff_solvable,
            'ap': plan,
            'error': err,
        }
        generated_ap_syntax_results.append(result)
        print(f"Generated: {cve_dir.name}/{config_dir.name}  syntax:{enhsp_ok}  solvable:{ff_solvable}")
        if plan:
            for i, action in enumerate(plan):
                print(f"  {i}: {action}")

Generated: CVE-2022-1471/reference  syntax:True  solvable:True
  0: ATTACKER-CRAFTS-MALICIOUS-JAVA-CLASS SEFA JAVA-GADGET-CLASS_SEFA COMMAND-EXECUTION-LOGIC_SEFA CVE_2022_1471
  1: ATTACKER-CRAFTS-MALICIOUS-YAML-PAYLOAD SEFA YAML-PAYLOAD_SEFA JAVA-GADGET-CLASS_SEFA JNDI-ENDPOINT_SEFA JAVA-GADGET-CLASS_SEFA CVE_2022_1471
  2: ATTACKER-SENDS-MALICIOUS-YAML-VIA-HTTP-POST HTTP-POST-REQUEST_SEFA SEFA YAML-PAYLOAD_SEFA YAML-ENDPOINT_SEFA CVE_2022_1471
  3: TARGET-SYSTEM-RECEIVES-HTTP-POST-REQUEST-WITH-MALICIOUS-PAYLOAD SEFA HTTP-POST-REQUEST_SEFA YAML-PAYLOAD_SEFA YAML-ENDPOINT_SEFA
  4: TARGET-SYSTEM-PASSES-YAML-TO-SNAKEYAML-LOADER HTTP-POST-REQUEST_SEFA YAML-PAYLOAD_SEFA SEFA SNAKEYAML-LIBRARY_SEFA
  5: TARGET-SYSTEM-STARTS-YAML-DESERIALIZATION-WITH-CONSTRUCTOR YAML-PAYLOAD_SEFA SNAKEYAML-LIBRARY_SEFA YAML-CONSTRUCTOR_SEFA SEFA
  6: TARGET-SYSTEM-INSTANTIATES-DANGEROUS-JAVA-GADGET-CLASS SEFA YAML-CONSTRUCTOR_SEFA JAVA-GADGET-CLASS_SEFA YAML-PAYLOAD_SEFA
  7: TARGET-SYSTEM-LOADS-MALICIOUS-C

#### 3.1.2 Extrinsic (future)

In [ ]:
# TODO: extrinsic syntactic evaluation

### 3.2 Semantic Evaluation

#### 3.2.1 Intrinsic

##### 3.2.1.1 Embedding: NL CVE Description vs PDDL Similarity

Block 1:  the function to compute the embedding similarity between the informal (natural language) specification and the PDDL code (domain + problem)

In [7]:
def embedding_similarity_intrinsic(descriptions, pddl_texts, model):
    """Cosine similarity matrix between NL descriptions and PDDL codes.
    Args:
        descriptions: list of NL description strings
        pddl_texts: list of PDDL (domain+problem) strings
        model: SentenceTransformer model
    Returns:
        numpy array of shape (len(descriptions), len(pddl_texts))
    """
    E_desc = model.encode(
        descriptions,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    E_pddl = model.encode(
        pddl_texts,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    S = E_desc @ E_pddl.T
    return S.float().cpu().numpy()

Block 2: call the function with the code and the specifications from the data set

In [8]:
descriptions = []
pddl_texts = []
labels = []

eval_set_path = Path(EVAL_SET_DIR)
for cve_dir in sorted(eval_set_path.iterdir()):
    if not cve_dir.is_dir():
        continue
    cve_id = cve_dir.name
    description = cve_descriptions.get(cve_id)
    if description is None:
        print(f"SKIP {cve_id}: no description in target_pool.json")
        continue
    for config_dir in sorted(cve_dir.iterdir()):
        if not config_dir.is_dir():
            continue
        domain_file = config_dir / DOMAIN_FILE
        problem_file = config_dir / PROBLEM_FILE
        if not domain_file.exists():
            continue
        domain = domain_file.read_text(encoding='utf-8').strip()
        problem = problem_file.read_text(encoding='utf-8').strip() if problem_file.exists() else ''
        descriptions.append(description)
        pddl_texts.append(domain + '\n' + problem)
        labels.append(f"{cve_id}/{config_dir.name}")

sim_matrix = embedding_similarity_intrinsic(descriptions, pddl_texts, embedding_model)
emb_intrinsic_results = []
for i, label in enumerate(labels):
    sim = float(sim_matrix[i, i])
    emb_intrinsic_results.append({'label': label, 'similarity': sim})
    print(f"{label}  similarity: {sim:.4f}")

CVE-2022-1471/reference  similarity: 0.3815
CVE-2025-66032/gpt4_1shot  similarity: 0.3689
CVE-2025-66032/qwen-max_1shot  similarity: 0.3575


##### 3.2.1.2 LLM as an Expert: NL CVE Description vs PDDL Match

Block 1: the function to prompt an LLM asking whether the NL specification and the PDDL code (domain + problem) are correctly matched

In [9]:
# TODO: improve the evaluation prompt
intrinsic_llm_template = prompt_env.get_template('llm_intrinsic.jinja')

def llm_eval_intrinsic(description, domain, problem, generator):
    """Ask LLM whether the PDDL code matches the NL description."""
    prompt = intrinsic_llm_template.render(
        description=description, domain=domain, problem=problem
    )
    messages = [{'role': 'user', 'content': prompt}]
    output = generator(messages)
    return output[0]['generated_text'][-1]['content']

 Block 2: call the function with the code and the specifications from the data set

In [10]:
llm_intrinsic_results = []
eval_set_path = Path(EVAL_SET_DIR)
for cve_dir in sorted(eval_set_path.iterdir()):
    if not cve_dir.is_dir():
        continue
    cve_id = cve_dir.name
    description = cve_descriptions.get(cve_id)
    if description is None:
        print(f"SKIP {cve_id}: no description in target_pool.json")
        continue
    for config_dir in sorted(cve_dir.iterdir()):
        if not config_dir.is_dir():
            continue
        domain_file = config_dir / DOMAIN_FILE
        problem_file = config_dir / PROBLEM_FILE
        if not domain_file.exists():
            continue
        domain = domain_file.read_text(encoding='utf-8').strip()
        problem = problem_file.read_text(encoding='utf-8').strip() if problem_file.exists() else ''
        response = llm_eval_intrinsic(description, domain, problem, llm)
        llm_intrinsic_results.append({
            'cve_id': cve_id,
            'config': config_dir.name,
            'llm_response': response,
        })
        print(f"{cve_id}/{config_dir.name}  response: {response[:200]}")

KeyboardInterrupt: 

Because waiting more than 200ms didn’t work, add the free Qwen model to quickly validate the pipeline.

In [ ]:
from openai import OpenAI

nvidia = OpenAI(
    api_key=os.getenv("NVIDIA_API_KEY"),
    base_url="https://integrate.api.nvidia.com/v1",
    timeout=120.0,
)
NVIDIA_MODEL = "meta/llama-3.3-70b-instruct"

llm_intrinsic_results_qwen = []
eval_set_path = Path(EVAL_SET_DIR)
for cve_dir in sorted(eval_set_path.iterdir()):
    if not cve_dir.is_dir():
        continue
    cve_id = cve_dir.name
    description = cve_descriptions.get(cve_id)
    if description is None:
        print(f"SKIP {cve_id}: no description in target_pool.json")
        continue
    for config_dir in sorted(cve_dir.iterdir()):
        if not config_dir.is_dir():
            continue
        domain_file = config_dir / DOMAIN_FILE
        problem_file = config_dir / PROBLEM_FILE
        if not domain_file.exists():
            continue
        domain = domain_file.read_text(encoding='utf-8').strip()
        problem = problem_file.read_text(encoding='utf-8').strip() if problem_file.exists() else ''
        prompt = intrinsic_llm_template.render(
            description=description, domain=domain, problem=problem
        )
        resp = nvidia.chat.completions.create(
            model=NVIDIA_MODEL,
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0.0,
            max_tokens=512,
            seed=SEED,
        )
        response = resp.choices[0].message.content
        llm_intrinsic_results_qwen.append({
            'cve_id': cve_id,
            'config': config_dir.name,
            'llm_response': response,
        })
        print(f"{cve_id}/{config_dir.name}  response: {response[:200]}")

#### 3.2.2 Extrinsic

##### 3.2.2.1 Embedding: reference PDDL vs generated PDDL Similarity

Block 1: the function to compute the embedding similarity between two blocks of PDDL code (domain + problem)

In [11]:
def embedding_similarity_extrinsic(pddl_texts_generated, pddl_texts_reference, model):
    """Cosine similarity matrix between generated and reference PDDL codes.
    Args:
        pddl_texts_generated: list of generated PDDL (domain+problem) strings
        pddl_texts_reference: list of reference PDDL (domain+problem) strings
        model: SentenceTransformer model
    Returns:
        numpy array of shape (len(generated), len(reference))
    """
    E_generated = model.encode(
        pddl_texts_generated,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    E_reference = model.encode(
        pddl_texts_reference,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    S = E_generated @ E_reference.T
    return S.float().cpu().numpy()

Block 2: call the function with the code from the data set

In [12]:
ref_by_cve = {entry['cve_id']: entry['attack_paths'] for entry in dataset}

emb_extrinsic_results = []
eval_set_path = Path(EVAL_SET_DIR)
for cve_dir in sorted(eval_set_path.iterdir()):
    if not cve_dir.is_dir():
        continue
    cve_id = cve_dir.name
    ref_aps = ref_by_cve.get(cve_id, [])
    if not ref_aps:
        print(f"SKIP {cve_id}: no reference APs in dataset")
        continue

    # Collect all generated PDDL texts for this CVE
    gen_texts = []
    gen_labels = []
    for config_dir in sorted(cve_dir.iterdir()):
        if not config_dir.is_dir():
            continue
        domain_file = config_dir / DOMAIN_FILE
        problem_file = config_dir / PROBLEM_FILE
        if not domain_file.exists():
            continue
        domain = domain_file.read_text(encoding='utf-8').strip()
        problem = problem_file.read_text(encoding='utf-8').strip() if problem_file.exists() else ''
        gen_texts.append(domain + '\n' + problem)
        gen_labels.append(config_dir.name)

    if not gen_texts:
        continue

    # Collect all reference PDDL texts for this CVE
    ref_texts = [ap['domain'] + '\n' + ap['problem'] for ap in ref_aps]
    ref_labels = [ap['ap_id'] for ap in ref_aps]

    # Batch compute similarity matrix: (num_generated x num_reference)
    sim_matrix = embedding_similarity_extrinsic(gen_texts, ref_texts, embedding_model)

    for i, gen_label in enumerate(gen_labels):
        for j, ref_label in enumerate(ref_labels):
            sim = float(sim_matrix[i, j])
            emb_extrinsic_results.append({
                'cve_id': cve_id,
                'generated': gen_label,
                'reference': ref_label,
                'similarity': sim,
            })
            print(f"{cve_id}/{gen_label} vs {ref_label}  similarity: {sim:.4f}")

CVE-2022-1471/reference vs AP1  similarity: 1.0000
SKIP CVE-2025-66032: no reference APs in dataset


##### 3.2.2.2 LLM as an Expert — Reference PDDL vs Candidate PDDL Match

Block 1: the function to prompt an LLM that given the informal (natural language) specification and the reference correct PDDL
code (domain + problem) if a second PDDL code (domain + problem) correctly matches the specification

In [15]:
# TODO: improve the evaluation prompt
extrinsic_llm_template = prompt_env.get_template('llm_extrinsic.jinja')

def llm_eval_extrinsic(description, domain_reference, problem_reference, domain_generated, problem_generated, generator):
    """Ask LLM whether generated PDDL matches the reference given the NL spec."""
    prompt = extrinsic_llm_template.render(
        description=description,
        domain_reference=domain_reference,
        problem_reference=problem_reference,
        domain_generated=domain_generated,
        problem_generated=problem_generated,
    )
    messages = [{'role': 'user', 'content': prompt}]
    output = generator(messages)
    return output[0]['generated_text'][-1]['content']

In [ ]:
ref_by_cve = {entry['cve_id']: entry['attack_paths'] for entry in dataset}

llm_extrinsic_results = []
eval_set_path = Path(EVAL_SET_DIR)
for cve_dir in sorted(eval_set_path.iterdir()):
    if not cve_dir.is_dir():
        continue
    cve_id = cve_dir.name
    description = cve_descriptions.get(cve_id)
    ref_aps = ref_by_cve.get(cve_id, [])
    if description is None or not ref_aps:
        print(f"SKIP {cve_id}: no description or no reference APs")
        continue
    for config_dir in sorted(cve_dir.iterdir()):
        if not config_dir.is_dir():
            continue
        domain_file = config_dir / DOMAIN_FILE
        problem_file = config_dir / PROBLEM_FILE
        if not domain_file.exists():
            continue
        domain_generated = domain_file.read_text(encoding='utf-8').strip()
        problem_generated = problem_file.read_text(encoding='utf-8').strip() if problem_file.exists() else ''
        for ref_ap in ref_aps:
            response = llm_eval_extrinsic(
                description,
                ref_ap['domain'], ref_ap['problem'],
                domain_generated, problem_generated,
                llm,
            )
            llm_extrinsic_results.append({
                'cve_id': cve_id,
                'generated': config_dir.name,
                'reference': ref_ap['ap_id'],
                'llm_response': response,
            })
            print(f"{cve_id}/{config_dir.name} vs {ref_ap['ap_id']}  response: {response[:200]}")

In [16]:
llm_extrinsic_results_qwen = []
eval_set_path = Path(EVAL_SET_DIR)
for cve_dir in sorted(eval_set_path.iterdir()):
    if not cve_dir.is_dir():
        continue
    cve_id = cve_dir.name
    description = cve_descriptions.get(cve_id)
    ref_aps = ref_by_cve.get(cve_id, [])
    if description is None or not ref_aps:
        print(f"SKIP {cve_id}: no description or no reference APs")
        continue
    for config_dir in sorted(cve_dir.iterdir()):
        if not config_dir.is_dir():
            continue
        domain_file = config_dir / DOMAIN_FILE
        problem_file = config_dir / PROBLEM_FILE
        if not domain_file.exists():
            continue
        domain_generated = domain_file.read_text(encoding='utf-8').strip()
        problem_generated = problem_file.read_text(encoding='utf-8').strip() if problem_file.exists() else ''
        for ref_ap in ref_aps:
            prompt = extrinsic_llm_template.render(
                description=description,
                domain_reference=ref_ap['domain'],
                problem_reference=ref_ap['problem'],
                domain_generated=domain_generated,
                problem_generated=problem_generated,
            )
            resp = nvidia.chat.completions.create(
                model=NVIDIA_MODEL,
                messages=[{'role': 'user', 'content': prompt}],
                temperature=0.0,
                max_tokens=512,
                seed=SEED,
            )
            response = resp.choices[0].message.content
            llm_extrinsic_results_qwen.append({
                'cve_id': cve_id,
                'generated': config_dir.name,
                'reference': ref_ap['ap_id'],
                'llm_response': response,
            })
            print(f"{cve_id}/{config_dir.name} vs {ref_ap['ap_id']}  response: {response[:200]}")

CVE-2022-1471/reference vs AP1  response: YES

The candidate PDDL code correctly models the same vulnerability as the reference. The domain and problem definitions in the candidate PDDL code match the reference PDDL code, and the actions and 
SKIP CVE-2025-66032: no description or no reference APs
